# 15 — Plan A: Current Condition of Existing Bridges

## Final architecture-aligned version

This notebook is the first new notebook after the established project workflow.

It does **not** search for, recreate, or retrain a missing 86-feature model.

It uses the actual frozen condition model produced by the project's model-freeze package.

### Project goal

Plan A:

```text
Existing bridge
      ↓
Current condition
      ↓
Future condition scenario
```

Plan B is deliberately **not implemented here**:

```text
New bridge location + design inputs
      ↓
Reference Library
      ↓
Similarity
      ↓
Top-K reference cohort
      ↓
Combined decision
      ↓
Recommended Bauwerksart
```

Notebook 15 only creates the current-condition layer required by both later workflows.

### Important model boundary

The frozen condition model uses:

```text
latitude
longitude
dtv
bauwerkstoff
bauwerksart
```

It does not use:

```text
laenge
breite
FEM
InfoCAD
structural design
```

Length and width remain available as reference-library / Plan-B similarity variables.

## Project paths

The project has exactly two working roots for the continuation:

```text
C:\Datenanalyse\final Project\Dataset_PlanA-B
C:\Datenanalyse\final Project\Output_PlanA-B
```

Notebook 15 consumes the canonical dataset and the already-frozen model package generated by the existing project workflow.

In [1]:
from pathlib import Path
import json
import hashlib
import warnings

import numpy as np
import pandas as pd
import joblib

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(r"C:\Datenanalyse\final Project")
DATASET_ROOT = PROJECT_ROOT / "Dataset_PlanA-B"
OUTPUT_ROOT = PROJECT_ROOT / "Output_PlanA-B"

OUTPUT_DIR = OUTPUT_ROOT / "15_Plan_A_Current_Bridge_Condition"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATASET_ROOT / "final_bridge_ml_dataset.parquet"

# Actual frozen package produced by the established project workflow.
MODEL_DIR = (
    OUTPUT_ROOT
    / "12_Final_ML_Model_Freeze_and_Packaging"
    / "model_package"
)

CONDITION_MODEL_PATH = MODEL_DIR / "condition_model.joblib"
MODEL_MANIFEST_PATH = MODEL_DIR / "model_manifest.json"

print("DATA_PATH:", DATA_PATH)
print("MODEL_DIR:", MODEL_DIR)
print("CONDITION_MODEL_PATH:", CONDITION_MODEL_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

for path, label in [
    (DATA_PATH, "canonical PlanA-B dataset"),
    (CONDITION_MODEL_PATH, "frozen condition model"),
]:
    if not path.exists():
        raise FileNotFoundError(
            f"{label} was not found at the established project path:\n{path}\n\n"
            "Notebook 15 does not create or retrain a replacement artifact."
        )

print("[PASS] configuration")

DATA_PATH: C:\Datenanalyse\final Project\Dataset_PlanA-B\final_bridge_ml_dataset.parquet
MODEL_DIR: C:\Datenanalyse\final Project\Output_PlanA-B\12_Final_ML_Model_Freeze_and_Packaging\model_package
CONDITION_MODEL_PATH: C:\Datenanalyse\final Project\Output_PlanA-B\12_Final_ML_Model_Freeze_and_Packaging\model_package\condition_model.joblib
OUTPUT_DIR: C:\Datenanalyse\final Project\Output_PlanA-B\15_Plan_A_Current_Bridge_Condition
[PASS] configuration


## 01 — Load canonical bridge-level dataset

The canonical input is:

```text
Dataset_PlanA-B\final_bridge_ml_dataset.parquet
```

The dataset is expected to represent one row per bridge.

In [2]:
df = pd.read_parquet(DATA_PATH)

print("Shape:", df.shape)

if df.shape != (52214, 97):
    raise ValueError(
        f"Expected canonical shape (52214, 97), received {df.shape}"
    )

if "bridge_id" not in df.columns:
    raise KeyError("bridge_id is missing.")

if not df["bridge_id"].is_unique:
    raise ValueError("bridge_id is not unique.")

if "zustandsnote" not in df.columns:
    raise KeyError("zustandsnote is missing.")

print("[PASS] 52,214 bridge records loaded")
print("[PASS] bridge_id is unique")
print("[PASS] zustandsnote is available as observed evidence")

Shape: (52214, 97)
[PASS] 52,214 bridge records loaded
[PASS] bridge_id is unique
[PASS] zustandsnote is available as observed evidence


## 02 — Resolve the model input columns

The established frozen condition-model contract is:

```text
latitude
longitude
dtv
bauwerkstoff
bauwerksart
```

Project mappings:

```text
dtv          ← traffic_dtv_mean
bauwerkstoff ← baustoffklasse
bauwerksart  ← bauwerksart_text
```

Latitude/longitude are used directly when present. If the canonical ML table does not contain them, the established project GIS representation `geom_x / geom_y` is transformed from EPSG:3857 to EPSG:4326.

In [3]:
def resolve_column(columns, candidates, label):
    for c in candidates:
        if c in columns:
            return c
    raise KeyError(
        f"Could not resolve required field '{label}'. "
        f"Candidates: {candidates}"
    )

COLS = {
    "bridge_id": "bridge_id",
    "dtv": resolve_column(
        df.columns,
        ["traffic_dtv_mean", "traffic_dtv_latest", "traffic_dtv_max", "dtv", "DTV"],
        "DTV"
    ),
    "bauwerkstoff": resolve_column(
        df.columns,
        ["baustoffklasse", "bauwerkstoff", "bauwerkstoff_text", "bridge_material", "material"],
        "Bauwerkstoff"
    ),
    "bauwerksart": resolve_column(
        df.columns,
        ["bauwerksart_text", "bauwerksart", "bridge_type", "type"],
        "Bauwerksart"
    ),
}

lat_candidates = ["latitude", "lat", "gis_latitude"]
lon_candidates = ["longitude", "lon", "lng", "gis_longitude"]

lat_col = next((c for c in lat_candidates if c in df.columns), None)
lon_col = next((c for c in lon_candidates if c in df.columns), None)

print("Resolved:")
for k, v in COLS.items():
    print(f"  {k}: {v}")
print("  latitude:", lat_col)
print("  longitude:", lon_col)

Resolved:
  bridge_id: bridge_id
  dtv: traffic_dtv_mean
  bauwerkstoff: baustoffklasse
  bauwerksart: bauwerksart_text
  latitude: None
  longitude: None


In [4]:
# Recover WGS84 coordinates from the established project geometry
# only if latitude/longitude are not already present.

if lat_col is None or lon_col is None:
    if "geom_x" not in df.columns or "geom_y" not in df.columns:
        raise KeyError(
            "No latitude/longitude fields and no geom_x/geom_y fields "
            "are available for the established coordinate conversion."
        )

    try:
        from pyproj import Transformer
    except ImportError as exc:
        raise ImportError(
            "pyproj is required for the established EPSG:3857 → EPSG:4326 conversion."
        ) from exc

    transformer = Transformer.from_crs(
        "EPSG:3857",
        "EPSG:4326",
        always_xy=True
    )

    x = pd.to_numeric(df["geom_x"], errors="coerce")
    y = pd.to_numeric(df["geom_y"], errors="coerce")

    lon_values, lat_values = transformer.transform(
        x.to_numpy(),
        y.to_numpy()
    )

    df["_plan_a_longitude"] = lon_values
    df["_plan_a_latitude"] = lat_values

    lat_col = "_plan_a_latitude"
    lon_col = "_plan_a_longitude"

print("[PASS] geographic input resolved")
print("Latitude source :", lat_col)
print("Longitude source:", lon_col)

[PASS] geographic input resolved
Latitude source : _plan_a_latitude
Longitude source: _plan_a_longitude


## 03 — Load and verify the frozen condition model

The model is consumed exactly as packaged by the established project workflow.

Expected artifact:

```text
Output_PlanA-B/
└── 12_Final_ML_Model_Freeze_and_Packaging/
    └── model_package/
        ├── bridge_type_classifier.joblib
        ├── condition_model.joblib
        └── model_manifest.json
```

No training operation occurs in Notebook 15.

In [5]:
condition_model = joblib.load(CONDITION_MODEL_PATH)

if not hasattr(condition_model, "predict"):
    raise TypeError(
        "condition_model.joblib does not provide predict()."
    )

print("Loaded condition model:", type(condition_model).__name__)

if MODEL_MANIFEST_PATH.exists():
    manifest = json.loads(
        MODEL_MANIFEST_PATH.read_text(encoding="utf-8")
    )

    expected_features = [
        "latitude",
        "longitude",
        "dtv",
        "bauwerkstoff",
        "bauwerksart",
    ]

    manifest_features = (
        manifest
        .get("condition_model", {})
        .get("features")
    )

    if manifest_features is not None:
        if manifest_features != expected_features:
            raise ValueError(
                "Frozen manifest condition-model features do not match "
                "the established contract.\n"
                f"Manifest: {manifest_features}\n"
                f"Expected: {expected_features}"
            )

    print("[PASS] frozen manifest verified")
else:
    print("[INFO] model_manifest.json not present; model artifact is still loadable.")

print("[PASS] frozen condition model loaded")

Loaded condition model: Pipeline
[PASS] frozen manifest verified
[PASS] frozen condition model loaded


## 04 — Build the exact condition-model input matrix

Observed bridge type is used for existing bridges.

This is important:

- Plan A evaluates **existing bridges**.
- Therefore `bauwerksart` is known from the bridge record.
- The bridge-type classifier is not needed for Plan A current-condition scoring.
- The classifier will be used later by Plan B for a proposed/new bridge.

In [6]:
condition_input = pd.DataFrame({
    "latitude": pd.to_numeric(df[lat_col], errors="coerce"),
    "longitude": pd.to_numeric(df[lon_col], errors="coerce"),
    "dtv": pd.to_numeric(df[COLS["dtv"]], errors="coerce"),
    "bauwerkstoff": df[COLS["bauwerkstoff"]].astype("string"),
    "bauwerksart": df[COLS["bauwerksart"]].astype("string"),
})

required_features = [
    "latitude",
    "longitude",
    "dtv",
    "bauwerkstoff",
    "bauwerksart",
]

if list(condition_input.columns) != required_features:
    raise ValueError("Condition-model feature order is incorrect.")

print("Condition input shape:", condition_input.shape)
print("[PASS] exact five-feature condition input constructed")

Condition input shape: (52214, 5)
[PASS] exact five-feature condition input constructed


## 05 — Data-quality gate before inference

The screenshot shows that the canonical dataset contains missing values in:

```text
latitude     786
longitude    786
dtv          21058
```

This is **not an error in the dataset** and must not stop Plan A.

The established frozen condition model contains its own preprocessing/imputation layer. Therefore Notebook 15 must pass the raw missing values to the frozen pipeline and must **not invent zeros or other replacement values**.

The gate below therefore checks:

- categorical inputs are present;
- missing numeric values are reported;
- the frozen model exposes the expected preprocessing/prediction interface.

The model itself remains unchanged.

In [7]:
# Missing numeric values are allowed because the frozen model package
# owns the preprocessing/imputation step.
#
# IMPORTANT:
# - Do NOT replace missing DTV with 0.
# - Do NOT replace missing coordinates with arbitrary coordinates.
# - Do NOT fit a new imputer in Notebook 15.
# - Pass the raw values to the frozen model pipeline.

numeric_cols = ["latitude", "longitude", "dtv"]

quality_report = pd.DataFrame({
    "field": condition_input.columns,
    "missing_count": [
        int(condition_input[c].isna().sum())
        for c in condition_input.columns
    ],
    "dtype": [
        str(condition_input[c].dtype)
        for c in condition_input.columns
    ],
})

display(quality_report)

categorical_missing = {
    c: int(condition_input[c].isna().sum())
    for c in ["bauwerkstoff", "bauwerksart"]
    if condition_input[c].isna().any()
}

if categorical_missing:
    raise ValueError(
        "Required categorical inference inputs contain missing values:\n"
        f"{categorical_missing}"
    )

print("[PASS] categorical inference inputs complete")
print()
print("Numeric missing values will be handled by the frozen model preprocessing:")
for c in numeric_cols:
    n = int(condition_input[c].isna().sum())
    print(f"  {c}: {n}")

print("[PASS] inference input quality gate")

,field,missing_count,dtype
0,latitude,786,float64
1,longitude,786,float64
2,dtv,21058,float64
3,bauwerkstoff,0,string
4,bauwerksart,0,string


[PASS] categorical inference inputs complete

Numeric missing values will be handled by the frozen model preprocessing:
  latitude: 786
  longitude: 786
  dtv: 21058
[PASS] inference input quality gate


## 06 — Current-condition inference

The frozen condition model predicts the current `zustandsnote` for every existing bridge.

The output is clipped to the established target scale [1, 4] only as a presentation guard; the model itself is not modified.

In [8]:
pred = np.asarray(
    condition_model.predict(condition_input[required_features]),
    dtype=float
)

if len(pred) != len(df):
    raise ValueError(
        f"Prediction count {len(pred)} != bridge count {len(df)}"
    )

if not np.isfinite(pred).all():
    raise ValueError("Predictions contain NaN or infinite values.")

pred_clipped = np.clip(pred, 1.0, 4.0)

result = df.copy()

result["zustandsnote_observed"] = result["zustandsnote"]
result["zustandsnote_predicted"] = pred_clipped

print("[PASS] predictions generated:", len(result))
print("Predicted range:", float(pred_clipped.min()), "to", float(pred_clipped.max()))

[PASS] predictions generated: 52214
Predicted range: 1.6609315253987043 to 2.6620224395731413


## 07 — Retrospective Plan A validation

Observed `zustandsnote` remains separate from the predicted value.

The comparison below is a retrospective full-population check. It is **not** a replacement for the independent validation of the frozen model.

In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_true = pd.to_numeric(
    result["zustandsnote_observed"],
    errors="coerce"
)

valid = y_true.notna()

if valid.sum() == 0:
    raise ValueError("No observed zustandsnote values are available.")

y_true_v = y_true.loc[valid]
y_pred_v = result.loc[valid, "zustandsnote_predicted"]

retrospective_mae = mean_absolute_error(y_true_v, y_pred_v)
retrospective_rmse = np.sqrt(
    mean_squared_error(y_true_v, y_pred_v)
)
retrospective_r2 = r2_score(y_true_v, y_pred_v)

print(f"Retrospective MAE : {retrospective_mae:.6f}")
print(f"Retrospective RMSE: {retrospective_rmse:.6f}")
print(f"Retrospective R²  : {retrospective_r2:.6f}")
print()
print("IMPORTANT: These are full-population retrospective metrics,")
print("not independent-test generalization metrics.")

Retrospective MAE : 0.313943
Retrospective RMSE: 0.421762
Retrospective R²  : 0.132001

IMPORTANT: These are full-population retrospective metrics,
not independent-test generalization metrics.


## 08 — Prepare the Plan A reference layer

The output intentionally preserves the fields required by Notebook 16 and later Plan B reference-library construction.

In particular, length, width, bridge type, material, DTV and geographic information are retained even though length/width are not inputs to the frozen condition model.

This is essential for the later Plan B similarity engine.

In [10]:
reference_columns = [
    "bridge_id",
    "bauwerksart_text",
    "baustoffklasse",
    "laenge",
    "breite",
    "traffic_dtv_mean",
    "zustandsnote_observed",
    "zustandsnote_predicted",
    "zustandsnotenklasse",
    "baujahr",
    "gis_ort",
    "gis_kreis",
    "gis_bundesland",
    "geom_x",
    "geom_y",
]

reference_columns = [
    c for c in reference_columns
    if c in result.columns
]

plan_a_current = result[reference_columns].copy()

# Standardized WGS84 fields for downstream Plan A / Plan B use.
plan_a_current["latitude"] = condition_input["latitude"].to_numpy()
plan_a_current["longitude"] = condition_input["longitude"].to_numpy()

print("Plan A current-condition/reference rows:", len(plan_a_current))
print("Columns:", len(plan_a_current.columns))

Plan A current-condition/reference rows: 52214
Columns: 17


## 09 — Export

```text
Output_PlanA-B/
└── 15_Plan_A_Current_Bridge_Condition/
    ├── plan_a_current_condition.parquet
    ├── plan_a_current_condition.csv
    ├── 15_plan_a_current_condition_summary.csv
    └── 15_plan_a_current_condition_manifest.json
```

This dataset is the direct input to Notebook 16.

In [11]:
PARQUET_OUT = OUTPUT_DIR / "plan_a_current_condition.parquet"
CSV_OUT = OUTPUT_DIR / "plan_a_current_condition.csv"
SUMMARY_OUT = OUTPUT_DIR / "15_plan_a_current_condition_summary.csv"
MANIFEST_OUT = OUTPUT_DIR / "15_plan_a_current_condition_manifest.json"

plan_a_current.to_parquet(PARQUET_OUT, index=False)
plan_a_current.to_csv(CSV_OUT, index=False)

summary = pd.DataFrame([
    {"metric": "bridge_count", "value": int(len(plan_a_current))},
    {"metric": "unique_bridge_id_count", "value": int(plan_a_current["bridge_id"].nunique())},
    {"metric": "observed_condition_count", "value": int(valid.sum())},
    {"metric": "retrospective_mae", "value": float(retrospective_mae)},
    {"metric": "retrospective_rmse", "value": float(retrospective_rmse)},
    {"metric": "retrospective_r2", "value": float(retrospective_r2)},
])

summary.to_csv(SUMMARY_OUT, index=False)

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

manifest_out = {
    "notebook": "15_Plan_A_Current_Bridge_Condition_FINAL",
    "status": "CURRENT_CONDITION_COMPLETE",
    "project_root": str(PROJECT_ROOT),
    "dataset_path": str(DATA_PATH),
    "dataset_sha256": sha256_file(DATA_PATH),
    "dataset_shape": list(df.shape),
    "bridge_count": int(len(df)),
    "condition_model_path": str(CONDITION_MODEL_PATH),
    "condition_model_sha256": sha256_file(CONDITION_MODEL_PATH),
    "condition_model_features": required_features,
    "target": "zustandsnote",
    "plan_a_role": "existing bridge current condition",
    "plan_b_role": "reference-library evidence only; similarity is downstream",
    "retrospective_metrics": {
        "MAE": float(retrospective_mae),
        "RMSE": float(retrospective_rmse),
        "R2": float(retrospective_r2),
    },
    "outputs": {
        "parquet": str(PARQUET_OUT),
        "csv": str(CSV_OUT),
        "summary": str(SUMMARY_OUT),
    },
}

MANIFEST_OUT.write_text(
    json.dumps(manifest_out, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("[PASS] Parquet:", PARQUET_OUT)
print("[PASS] CSV:", CSV_OUT)
print("[PASS] Summary:", SUMMARY_OUT)
print("[PASS] Manifest:", MANIFEST_OUT)

[PASS] Parquet: C:\Datenanalyse\final Project\Output_PlanA-B\15_Plan_A_Current_Bridge_Condition\plan_a_current_condition.parquet
[PASS] CSV: C:\Datenanalyse\final Project\Output_PlanA-B\15_Plan_A_Current_Bridge_Condition\plan_a_current_condition.csv
[PASS] Summary: C:\Datenanalyse\final Project\Output_PlanA-B\15_Plan_A_Current_Bridge_Condition\15_plan_a_current_condition_summary.csv
[PASS] Manifest: C:\Datenanalyse\final Project\Output_PlanA-B\15_Plan_A_Current_Bridge_Condition\15_plan_a_current_condition_manifest.json


## 10 — Final gate and handoff

Notebook 15 is complete only if:

- 52,214 bridges are present;
- bridge IDs are unique;
- current predictions exist for all bridges;
- output files exist;
- the frozen condition model was used.

### Handoff

Notebook 16 must consume:

```text
plan_a_current_condition.parquet
```

Notebook 16 will add the separate temporal/cohort layer.

It must not retrain this current-condition model.

Plan B will later use the resulting Plan A reference layer to calculate:

```text
Bauwerksart 30%
Bauwerkstoff 20%
Length 15%
Width 10%
DTV 15%
Geographic proximity 10%
```

and then construct the Top-K reference cohort and combined decision layer.

The weights are the established transparent project contract, not statistically optimized weights.

In [12]:
final_checks = {
    "bridge_count_52214": len(plan_a_current) == 52214,
    "unique_bridge_id": plan_a_current["bridge_id"].nunique() == 52214,
    "prediction_count": len(plan_a_current["zustandsnote_predicted"]) == 52214,
    "finite_predictions": np.isfinite(
        plan_a_current["zustandsnote_predicted"].to_numpy()
    ).all(),
    "parquet_exists": PARQUET_OUT.exists(),
    "csv_exists": CSV_OUT.exists(),
    "summary_exists": SUMMARY_OUT.exists(),
    "manifest_exists": MANIFEST_OUT.exists(),
}

print("FINAL NOTEBOOK 15 GATE")
for key, passed in final_checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {key}")

if not all(final_checks.values()):
    raise RuntimeError("Notebook 15 final gate failed.")

print("\n15 STATUS: COMPLETE")
print("Next notebook input:", PARQUET_OUT)

FINAL NOTEBOOK 15 GATE
[PASS] bridge_count_52214
[PASS] unique_bridge_id
[PASS] prediction_count
[PASS] finite_predictions
[PASS] parquet_exists
[PASS] csv_exists
[PASS] summary_exists
[PASS] manifest_exists

15 STATUS: COMPLETE
Next notebook input: C:\Datenanalyse\final Project\Output_PlanA-B\15_Plan_A_Current_Bridge_Condition\plan_a_current_condition.parquet
